# Model pref chosen



# Importing packages

We first import some useful Python packages. pandas is an open source data analysis and manipulation tool built in the Python programming language. Biogeme is a Python package for estimating discrete choice models to explain and predict how people choose among alternatives.

In [1]:
import pandas as pd
import biogeme.database as db
import biogeme.biogeme as bio
from tqdm.notebook import tqdm
from biogeme.expressions import Beta, Variable, log, exp
from biogeme import models
from biogeme.results_processing import EstimationResults
from biogeme.results_processing import get_pandas_estimated_parameters, html_output

g++ not available, if using conda: `conda install gxx`
c:\Users\chloe\Desktop\3A\Data Science\Modelling Choice Behavior\Biogeme\myvenv\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


# Reading and preparing the data

We first import the data into Pandas. We read the data from a text file, where the data are separated by tabs. 

In [2]:
df = pd.read_csv('lpmc19.dat', sep='\t')
df

,trip_id,household_id,person_n,trip_n,travel_mode,purpose,fueltype,faretype,bus_scale,survey_year,...,dur_pt_access,dur_pt_rail,dur_pt_bus,dur_pt_int,pt_interchanges,dur_driving,cost_transit,cost_driving_fuel,cost_driving_ccharge,driving_traffic_percent
0,5,0,1,3,4,3,1,4,1.0,1,...,0.205556,0.000000,0.221111,0.000000,0,0.222222,1.5,0.77,0.0,0.097500
1,16,2,1,2,4,3,1,1,1.0,1,...,0.098611,0.000000,0.058333,0.000000,0,0.058611,1.5,0.16,0.0,0.123223
2,26,7,0,2,1,3,2,1,1.0,1,...,0.068333,0.000000,0.033889,0.000000,0,0.055278,1.5,0.10,0.0,0.105528
3,27,7,1,0,4,4,2,1,1.0,1,...,0.193889,0.516667,0.000000,0.100000,1,0.340833,1.5,2.04,0.0,0.280359
4,52,12,1,2,4,5,2,1,1.0,1,...,0.062500,0.000000,0.491944,0.094722,1,0.355556,3.0,1.19,0.0,0.249219
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,81040,17604,2,0,3,1,1,2,0.0,3,...,0.264444,0.000000,0.353333,0.176667,1,0.288889,0.0,0.85,0.0,0.175000
4996,81059,17606,1,2,3,3,6,1,1.0,3,...,0.100278,0.000000,0.360278,0.000000,0,0.281667,1.5,0.63,0.0,0.384615
4997,81062,17607,0,4,3,5,6,1,1.0,3,...,0.069444,0.000000,0.166389,0.000000,0,0.146111,1.5,0.28,0.0,0.653992
4998,81071,17610,0,0,4,3,2,1,1.0,3,...,0.206111,0.000000,0.400000,0.000000,0,0.259167,1.5,0.88,0.0,0.233655


In [3]:
df['age'].dropna()


0       54
1       41
2       49
3       51
4       56
        ..
4995    17
4996    33
4997    38
4998    33
4999    40
Name: age, Length: 5000, dtype: int64

We then import this database into Biogeme.

In [4]:
database = db.Database('netherlands', df)

We identify the columns that will be used as variables in our model. 

In [5]:
trip_id = Variable('trip_id')
travel_mode = Variable('travel_mode')
purpose = Variable('purpose')
distance = Variable('distance')
dur_walking = Variable('dur_walking')
dur_cycling = Variable('dur_cycling')
dur_pt_access = Variable('dur_pt_access')
dur_pt_rail  = Variable('dur_pt_rail')
dur_pt_bus  = Variable('dur_pt_bus')
dur_pt_int = Variable('dur_pt_int')
dur_driving = Variable('dur_driving')
cost_transit = Variable('cost_transit')
cost_driving_fuel = Variable('cost_driving_fuel')
cost_driving_ccharge = Variable('cost_driving_ccharge')
driving_traffic_percent = Variable('driving_traffic_percent')
age = Variable('age')

In [6]:
database.num_rows()

5000

We define the total travel time by public transportation

In [7]:
dur_pt_total = dur_pt_access + dur_pt_rail + dur_pt_bus + dur_pt_int
cost_car = cost_driving_fuel + cost_driving_ccharge

We convert the Guilders into Euros.

In [8]:
POUNDS_TO_EUROS = 1.16
cost_car_euro = cost_car * POUNDS_TO_EUROS
cost_pt_euro = cost_transit * POUNDS_TO_EUROS

# Model specification

We can now specify a model that contains 4 ASC

In [9]:
asc_car = Beta('asc_car', 0, None, None, 0)
asc_walk = Beta('asc_walk', 0, None, None, 0)
asc_bike = Beta('asc_bike', 0, None, None, 0)
asc_pt = Beta('asc_pt', 0, None, None, 0)
beta_cost = Beta('beta_cost_car', 0, None, None, 0)
beta_age = Beta('beta_age', 0, None, None, 0)
beta_time_car = Beta('beta_time_car', 0, None, None, 0)
beta_time_walk = Beta('beta_time_walk', 0, None, None, 0)
beta_time_bike = Beta('beta_time_bike', 0, None, None, 0)
beta_time_pt = Beta('beta_time_pt', 0, None, None, 0)
beta_driving_traffic_percent = Beta('beta_driving_traffic_percent', 0, None, None, 0)

We  write the utility functions:
**We choose walking as the reference**

In [10]:
v_car = asc_car + beta_age * age + beta_cost * cost_car_euro + beta_time_car * log(dur_driving) + beta_driving_traffic_percent * driving_traffic_percent
v_walk = 0 + beta_time_walk * log(dur_walking)
v_bike = asc_bike + beta_age * age + beta_time_bike * log(dur_cycling)
v_pt = asc_pt + beta_age * age + beta_cost * cost_pt_euro + beta_time_pt * log(dur_pt_total)

And we write the choice model:

In [11]:
V = {1: v_walk, 2: v_bike, 3: v_pt, 4: v_car}

Blabla

In [12]:
logprob = models.loglogit(V, None, travel_mode)

We initialize Biogeme with this expression and the database. We call the model 'binary_netherlands'.

In [13]:
biogeme = bio.BIOGEME(database, logprob)
biogeme.model_name = 'model_3'

# Model estimation

Biogeme tries to read a file `__binary_netherlands.iter` containing intermediary results from a previous estimation run. If it does not find it, it triggers a warning that can be ignored.

In [14]:
results_model_3 = biogeme.estimate()

We print some general statistics: 

In [15]:
print(results_model_3.print_general_statistics())

Number of estimated parameters             10
Sample size                                5000
Excluded observations                      0
Init log likelihood                        -6931.472
Final log likelihood                       -4121.26
Likelihood ratio test for the init. model  5620.423
Rho-square for the init. model             0.405
Rho-square-bar for the init. model         0.404
Akaike Information Criterion               8262.52
Bayesian Information Criterion             8327.692
Final gradient norm                        1.7638E-02
Bootstrapping time                         None


We print the estimation results

In [16]:
get_pandas_estimated_parameters(estimation_results=results_model_3)

{'Estimated parameters':                            Name     Value  Robust std err.  Robust t-stat.  \
 0                beta_time_walk -4.191312         0.127085      -32.980393   
 1                      asc_bike -2.575308         0.240312      -10.716541   
 2                      beta_age  0.011037         0.002456        4.494215   
 3                beta_time_bike -2.113964         0.130568      -16.190541   
 4                        asc_pt  1.403750         0.157965        8.886445   
 5                 beta_cost_car -0.122815         0.014454       -8.497253   
 6                  beta_time_pt -1.676739         0.117366      -14.286451   
 7                       asc_car  1.740407         0.242987        7.162548   
 8                 beta_time_car -1.542057         0.110661      -13.934966   
 9  beta_driving_traffic_percent -2.695289         0.246242      -10.945708   
 
    Robust p-value  
 0    0.000000e+00  
 1    0.000000e+00  
 2    6.982700e-06  
 3    0.000000e+00  


---
## Market shares — [2.5 points]

Assume that stratified random sampling was used to produce your sample.
The sample was divided into strata (i.e., groups) and simple random
sampling was used to sample observations in each stratum. We consider the
following strata:

- **S1:** females aged 40 years or younger
- **S2:** females aged 41 years or older
- **S3:** males aged 40 years or younger
- **S4:** males aged 41 years or older

London population estimates in 2015 (Source: ONS):

| | Age ≤ 40 | Age > 40 |
|---|---|---|
| Male | 2'676'249 | 1'633'263 |
| Female | 2'599'058 | 1'765'143 |

## Setup

In [17]:
import numpy as np
import pandas as pd
from scipy import stats

import biogeme.database as db
import biogeme.biogeme as bio
from biogeme import models
from biogeme.expressions import Beta, Variable
from biogeme.results_processing import EstimationResults

# Load the LPMC sample
data = pd.read_csv("lpmc19.dat", sep="\t")
data.head()

,trip_id,household_id,person_n,trip_n,travel_mode,purpose,fueltype,faretype,bus_scale,survey_year,...,dur_pt_access,dur_pt_rail,dur_pt_bus,dur_pt_int,pt_interchanges,dur_driving,cost_transit,cost_driving_fuel,cost_driving_ccharge,driving_traffic_percent
0,5,0,1,3,4,3,1,4,1.0,1,...,0.205556,0.000000,0.221111,0.000000,0,0.222222,1.5,0.77,0.0,0.097500
1,16,2,1,2,4,3,1,1,1.0,1,...,0.098611,0.000000,0.058333,0.000000,0,0.058611,1.5,0.16,0.0,0.123223
2,26,7,0,2,1,3,2,1,1.0,1,...,0.068333,0.000000,0.033889,0.000000,0,0.055278,1.5,0.10,0.0,0.105528
3,27,7,1,0,4,4,2,1,1.0,1,...,0.193889,0.516667,0.000000,0.100000,1,0.340833,1.5,2.04,0.0,0.280359
4,52,12,1,2,4,5,2,1,1.0,1,...,0.062500,0.000000,0.491944,0.094722,1,0.355556,3.0,1.19,0.0,0.249219


**1.** Report the size and weight of each stratum in your sample.
**[1 point]**

In [18]:
N3 = 2676249
N4 = 1633263
N1 = 2599058
N2 = 1765143

#Population
N = N1 + N2 + N3 + N4

#Table of Ng/N
table_ng_n = [[N3/N, N4/N], [N1/N, N2/N]] 

table = pd.DataFrame(table_ng_n, columns=["Age =< 40", "Age > 40"], index=["Male", "Female"])
print(f'Table of Ng/N:')
print(table)

Table of Ng/N:
        Age =< 40  Age > 40
Male     0.308547  0.188300
Female   0.299648  0.203505


In [19]:
S3 = len(data.loc[(data['age'] <= 40) & (data['female'] == 0)])
S4 = len(data.loc[(data['age'] > 40) & (data['female'] == 0)])
S1 = len(data.loc[(data['age'] <= 40) & (data['female'] == 1)])
S2 = len(data.loc[(data['age'] > 40) & (data['female'] == 1)])

#Population of the sample
S = S1 + S2 + S3 + S4

#Table of Sg/S
table_sg_s = [[S3/S, S4/S], [S1/S, S2/S]]

table = pd.DataFrame(table_sg_s, columns=["Age =< 40", "Age > 40"], index=["Male", "Female"])
print(f'Table of Sg/S:')
print(table)

Table of Sg/S:
        Age =< 40  Age > 40
Male       0.2502    0.2278
Female     0.2830    0.2390


In [20]:
#Table of weights Ng/N * S/Sg
table_weights_strata = [[table_ng_n[0][0]*(1/table_sg_s[0][0]),table_ng_n[0][1]*(1/table_sg_s[0][1])], [table_ng_n[1][0]*(1/table_sg_s[1][0]),table_ng_n[1][1]*(1/table_sg_s[1][1])]]

table_weights = pd.DataFrame(table_weights_strata, columns=["Age =< 40", "Age > 40"], index=["Male", "Female"])
print(f'Table of weights:')
print(table_weights)
print(f'Size of stratum Male and under 40:', S3)
print(f'Size of stratum Male and over 40:', S4)
print(f'Size of stratum Female and under 40:', S1)
print(f'Size of stratum Female and over 40:', S2)

Table of weights:
        Age =< 40  Age > 40
Male     1.233202  0.826604
Female   1.058826  0.851485
Size of stratum Male and under 40: 1251
Size of stratum Male and over 40: 1139
Size of stratum Female and under 40: 1415
Size of stratum Female and over 40: 1195


In [21]:
# Your answer here: stratum sizes and weights (S1-S4)

**2.** Using $\text{Model}_{\text{pref}}$ and the weights of your strata,
compute the predicted market share of each mode and their confidence
intervals. Do the obtained results match your expectations? **[1 point]**

In [ ]:
#Création colonne avec weights pour chaque individu, la somme de cette colonne doit etre egale à la somme du nombre d'observation du sample
df['Weight'] = np.select(
    [(df['female'] == 0) & (df['age'] <= 40),
     (df['female'] == 0) & (df['age'] >  40),
     (df['female'] == 1) & (df['age'] <= 40),
     (df['female'] == 1) & (df['age'] >  40)],
    [table_weights_strata[0][0], table_weights_strata[0][1],
     table_weights_strata[1][0], table_weights_strata[1][1]])

database = db.Database('with weights', df)

In [ ]:
#Verification
sum_weights = df['Weight'].sum()
print(sum_weights, len(df))

5000.0 5000


In [ ]:
#Ici on ajoute Weight comme variable à simuler
Weight = Variable('Weight')

av = {1: 1, 2: 1, 3: 1, 4: 1}

simulate = {
    'Weight': Weight,
    'walk':  models.logit(V, av, 1),
    'cycle': models.logit(V, av, 2),
    'pt':    models.logit(V, av, 3),
    'drive': models.logit(V, av, 4)
}

#Prédit pour chaque individu la proba qu'il/elle choisse l'alternative
biosim = bio.BIOGEME(database, simulate)
simulated_values = biosim.simulate(results_model_3.get_beta_values())

In [31]:
simulated_values

,Weight,walk,cycle,pt,drive
0,0.826604,0.008567,0.010502,0.250706,0.730225
1,0.826604,0.516803,0.027677,0.085510,0.370011
2,0.851485,0.735403,0.019572,0.071201,0.173825
3,0.826604,0.000141,0.004075,0.306229,0.689555
4,0.851485,0.002485,0.011901,0.307055,0.678558
...,...,...,...,...,...
4995,1.233202,0.019427,0.024233,0.209220,0.747120
4996,1.058826,0.050954,0.042747,0.435536,0.470762
4997,1.058826,0.437426,0.081696,0.322224,0.158654
4998,1.058826,0.004127,0.010567,0.257711,0.727594


In [ ]:
#The market shares are calculated using the weighted mean of the individual probabilities
simulated_values['Weighted walk'] = (
    simulated_values['Weight'] * 
    simulated_values['walk']
)
simulated_values['Weighted cycle'] = (
    simulated_values['Weight'] * 
    simulated_values['cycle']
)
simulated_values['Weighted pt'] = (
    simulated_values['Weight'] * 
    simulated_values['pt']
)
simulated_values['Weighted drive'] = (
    simulated_values['Weight'] * 
    simulated_values['drive']
)

In [37]:
market_share_walk = simulated_values['Weighted walk'].mean()
print(f'Market share for walk: {100*market_share_walk:.1f}%')

market_share_cycle = simulated_values['Weighted cycle'].mean()
print(f'Market share for cycle: {100*market_share_cycle:.1f}%')

market_share_pt = simulated_values['Weighted pt'].mean()
print(f'Market share for pt: {100*market_share_pt:.1f}%')

market_share_drive = simulated_values['Weighted drive'].mean()
print(f'Market share for drive: {100*market_share_drive:.1f}%')

Market share for walk: 18.8%
Market share for cycle: 2.9%
Market share for pt: 35.1%
Market share for drive: 43.2%


**We calculate now the confidence intervals of the market shares**

In [38]:
results_bootstrapping = biogeme.estimate(run_bootstrap=True)
estimated_parameters = get_pandas_estimated_parameters(estimation_results=results_bootstrapping)
display(estimated_parameters)

Bootstraps:   0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Bootstraps:   0%|          | 0/100 [00:47<?, ?it/s]


{'Estimated parameters':                            Name     Value  Bootstrap std err.  \
 0                beta_time_walk -4.191312            0.135972   
 1                      asc_bike -2.575308            0.213817   
 2                      beta_age  0.011037            0.002594   
 3                beta_time_bike -2.113964            0.119995   
 4                        asc_pt  1.403750            0.139387   
 5                 beta_cost_car -0.122815            0.014999   
 6                  beta_time_pt -1.676739            0.116345   
 7                       asc_car  1.740407            0.219080   
 8                 beta_time_car -1.542057            0.105198   
 9  beta_driving_traffic_percent -2.695289            0.243777   
 
    Bootstrap t-stat.  Bootstrap p-value  
 0         -30.824905       0.000000e+00  
 1         -12.044435       0.000000e+00  
 2           4.255325       2.087457e-05  
 3         -17.617157       0.000000e+00  
 4          10.070893       0.000

We obtain a sample of values for the parameters we can use to calculate the 90% confidence intervals on the simulated quantities.

In [39]:
betas = biogeme.free_betas_names
b = results_bootstrapping.get_betas_for_sensitivity_analysis(betas, size = 100)

left, right = biosim.confidence_intervals(b, interval_size=0.9)

100%|██████████| 100/100 [00:00<00:00, 379.05it/s]


In [43]:
left['Weighted walk'] = (
    left['Weight'] * 
    left['walk']
)
left['Weighted cycle'] = (
    left['Weight'] * 
    left['cycle']
)
left['Weighted pt'] = (
    left['Weight'] * 
    left['pt']
)
left['Weighted drive'] = (
    left['Weight'] * 
    left['drive']
)

right['Weighted walk'] = (
    right['Weight'] * 
    right['walk']
)
right['Weighted cycle'] = (
    right['Weight'] * 
    right['cycle']
)
right['Weighted pt'] = (
    right['Weight'] * 
    right['pt']
)
right['Weighted drive'] = (
    right['Weight'] * 
    right['drive']
)

In [44]:
left_market_share_walk = left['Weighted walk'].mean()
left_market_share_cycle = left['Weighted cycle'].mean()
left_market_share_pt = left['Weighted pt'].mean()
left_market_share_drive = left['Weighted drive'].mean()

right_market_share_walk = right['Weighted walk'].mean()
right_market_share_cycle = right['Weighted cycle'].mean()
right_market_share_pt = right['Weighted pt'].mean()
right_market_share_drive = right['Weighted drive'].mean()

In [47]:
print(
    f'Market share for walk: {100*market_share_walk:.1f}% '
    f'CI: ['
    f'{100*left_market_share_walk:.1f}%-'
    f'{100*right_market_share_walk:.1f}%'
    f']'
)
print(
    f'Market share for cycle: {100*market_share_cycle:.1f}% '
    f'CI: ['
    f'{100*left_market_share_cycle:.1f}%-'
    f'{100*right_market_share_cycle:.1f}%'
    f']'
)
print(
    f'Market share for pt: {100*market_share_pt:.1f}% '
    f'CI: ['
    f'{100*left_market_share_pt:.1f}%-'
    f'{100*right_market_share_pt:.1f}%'
    f']'
)
print(
    f'Market share for drive: {100*market_share_drive:.1f}% '
    f'CI: ['
    f'{100*left_market_share_drive:.1f}%-'
    f'{100*right_market_share_drive:.1f}%'
    f']'
)

Market share for walk: 18.8% CI: [17.5%-20.0%]
Market share for cycle: 2.9% CI: [2.4%-3.4%]
Market share for pt: 35.1% CI: [33.2%-37.0%]
Market share for drive: 43.2% CI: [41.1%-45.4%]


In [ ]:
# Your answer here: weighted predicted market shares + confidence intervals

**3.** Compare the market shares predicted by $\text{Model}_{\text{pref}}$
with the weighted market shares computed using the actual choices.
**[0.5 point]**

In [ ]:
# Your answer here